# Rotten Fruit Detector — SageMaker Studio Training (KerasHub RetinaNet)

TensorFlow-native object detection with **KerasHub RetinaNet** — no TF Object Detection API, no protobuf/protoc setup. `main()` reads the Roboflow CSV export into a tf.data pipeline, fine-tunes RetinaNet with `model.fit()`, evaluates, and saves the model.

Run from inside the cloned repo on a **GPU** kernel. Expected `dataset.zip` contents:

```text
train/_annotations.csv + *.jpg
valid/_annotations.csv + *.jpg
test/_annotations.csv  + *.jpg
```
(Roboflow **TensorFlow Object Detection** CSV export.) Split folders may sit at the zip root or one level deep — the notebook auto-detects.

Smoke-test with `epochs=1` first; stop the kernel when done to cap cost.

In [ ]:
# 1. Install deps (TensorFlow + KerasHub)
%pip install -q -r ../requirements.txt

In [ ]:
# 2. Download the dataset from S3, unzip it, and locate the split folders
import zipfile
from pathlib import Path

import boto3

S3_BUCKET = "YOUR_BUCKET"
S3_KEY = "datasets/dataset.zip"
PROJECT_DIR = Path.cwd().parent

zip_path = PROJECT_DIR / "dataset.zip"
boto3.client("s3").download_file(S3_BUCKET, S3_KEY, str(zip_path))

EXTRACT_DIR = PROJECT_DIR / "dataset"
with zipfile.ZipFile(zip_path) as archive:
    archive.extractall(EXTRACT_DIR)

def find_dataset_root(base):
    base = Path(base)
    for path in [base, *(p for p in base.rglob("*") if p.is_dir())]:
        if (path / "train").is_dir():
            return path
    raise FileNotFoundError(f"No 'train' folder found under {base}")

DATA_DIR = find_dataset_root(EXTRACT_DIR)
print("Dataset root:", DATA_DIR)
print("Contains:", sorted(p.name for p in DATA_DIR.iterdir() if p.is_dir()))

In [ ]:
# 3. Train -> evaluate -> save (KerasHub uses the TensorFlow backend)
import os
import sys

os.environ["KERAS_BACKEND"] = "tensorflow"  # set BEFORE importing keras/keras_hub
os.chdir(PROJECT_DIR)
sys.path.insert(0, str(PROJECT_DIR / "src"))

from train import main

# Faster run (~<1 hr on a single T4). For a hard time cap on large datasets,
# add steps_per_epoch=200 (each epoch then trains on 200 batches only).
model = main(data_dir=DATA_DIR, epochs=10, batch_size=4, image_size=512)

In [ ]:
# 4. Upload the saved model to S3
import sagemaker

session = sagemaker.Session()
uri = session.upload_data(
    path="retinanet_fruit.keras",
    bucket=session.default_bucket(),
    key_prefix="rotten-fruit-detection/retinanet",
)
print("Uploaded to:", uri)